<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
경량 LLM 질의응답 & 파인튜닝
</div>

In [ ]:
%%capture
%pip install -U unsloth

In [ ]:
import unsloth
import torch

In [ ]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
device

In [ ]:
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU 총 메모리: {total:.1f} GB")

# 4비트 Qwen2.5 모델 로딩

In [ ]:
from unsloth import FastLanguageModel

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
# 모델을 불러온 뒤 GPU 메모리를 확인해 봅시다
used = torch.cuda.memory_allocated() / 1024**3
print(f"현재 GPU 메모리 사용량: {used:.2f} GB")

# 질의응답 해보기

In [ ]:
from transformers import TextStreamer

In [ ]:
def ask(question, max_length=1024):
    """모델에게 질문하고 답변을 실시간(스트리밍)으로 출력하는 함수"""
    messages = [
        {"role": "user", "content": question},
    ]

    # 채팅 템플릿 적용: 사람이 읽는 문장 → 모델이 이해하는 형식
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,   # "이제 assistant가 답할 차례" 표시 추가
        return_tensors        = "pt",
    ).to("cuda")

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)  # 답변만 실시간 출력
    _ = model.generate(
        input_ids          = inputs,
        streamer           = streamer,
        max_length         = max_length,
        repetition_penalty = 1.05,
        temperature        = 0.7,   # 낮을수록 일관된 답, 높을수록 창의적인 답
        top_p              = 0.9,
        do_sample          = True,
    )

In [ ]:
# 추론 모드로 전환 (2배 빠른 추론)
FastLanguageModel.for_inference(model)

In [ ]:
ask("스웨터의 유래에 대해서 알려주세요.")

In [ ]:
ask("인공지능과 머신러닝의 차이를 초등학생에게 설명하듯이 해 주세요.")

# LoRA 어댑터 붙이기

## 💡 LoRA (Low-Rank Adaptation)란?
전체 모델(5억 개 파라미터)을 다 학습하면 메모리가 엄청나게 필요합니다.
LoRA는 원본 모델은 **얼음처럼 얼려두고(freeze)**, 옆에 **아주 작은 어댑터 행렬**만 붙여서 그것만 학습합니다.

```
원본 가중치 W (고정) + 작은 어댑터 A×B (학습) = 새로운 능력
```

- 학습 파라미터가 전체의 **1% 수준**으로 줄어듭니다
- 메모리 사용량이 크게 줄어 무료 GPU에서도 파인튜닝이 가능해집니다
- `r`(rank) 값이 클수록 표현력 ↑, 메모리 사용 ↑


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # LoRA rank: 어댑터의 크기 (8, 16, 32 등)
    target_modules = [         # 어댑터를 붙일 위치 (어텐션 + MLP 층)
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha    = 16,        # 어댑터 출력의 스케일 (보통 r과 같게 설정)
    lora_dropout  = 0,         # 0이면 Unsloth가 최적화된 경로 사용
    bias          = "none",
    use_gradient_checkpointing = "unsloth",  # 메모리 추가 절약 (긴 문맥 대응)
    random_state  = 42,
)

In [ ]:
# 전체 파라미터 중 실제로 학습되는 비율을 확인해 봅시다
model.print_trainable_parameters()

# 학습 데이터 준비 (KoAlpaca)

**KoAlpaca**는 한국어 질문-답변 쌍으로 이루어진 공개 데이터셋입니다.
교육 시간을 아끼기 위해 **1,000건만** 사용합니다.

각 데이터를 모델의 채팅 템플릿 형식으로 변환합니다:
```
사용자 질문 → <|im_start|>user ... <|im_end|>
모범 답변   → <|im_start|>assistant ... <|im_end|>
```


In [ ]:
from datasets import load_dataset

# KoAlpaca 데이터셋에서 1,000건만 로드
dataset = load_dataset("beomi/KoAlpaca-v1.1a", split="train[:1000]")

print("데이터 개수:", len(dataset))

In [ ]:
dataset[1]

In [ ]:
print("[+] 질문:\n", dataset[0]["instruction"])
print("[+] 답변:\n", dataset[0]["output"])

In [ ]:
def to_chat_format(example):
    """질문/답변 쌍을 모델의 채팅 템플릿 텍스트로 변환"""
    messages = [
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,               # 학습용이므로 텍스트 그대로 반환
        add_generation_prompt = False,  # 답변을 위한 assistant는 학습을 위해 만들기 때문에 False
    )
    return {"text": text}

In [ ]:
dataset = dataset.map(to_chat_format)

In [ ]:
dataset

In [ ]:
print(dataset[0]["text"])

# 파인튜닝 실행 (SFT)

**SFT(Supervised Fine-Tuning)**: 질문-모범답변 쌍을 보여주며 "이렇게 답해"라고 가르치는 학습입니다.

### ⚙️ 주요 설정 설명
| 설정 | 값 | 의미 |
|---|---|---|
| `max_steps` | 60 | 학습 스텝 수 (교육용으로 짧게) |
| `per_device_train_batch_size` | 2 | 한 번에 처리할 데이터 수 |
| `gradient_accumulation_steps` | 4 | 4번 모아서 업데이트 → 실질 배치 8 |
| `learning_rate` | 2e-4 | 학습 속도 |

60스텝 × 배치 8 = 총 480개 샘플을 학습합니다


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    args = SFTConfig(
        dataset_text_field = "text",   # 학습에 사용할 컬럼 이름
        max_seq_length     = 2048,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps          = 60,       # ⏱️ 교육용 짧은 학습
        warmup_steps       = 10,
        learning_rate      = 2e-4,
        logging_steps      = 10,        # 10스텝마다 loss 출력
        optim              = "adamw_8bit",  # 8bit 옵티마이저 → 메모리 절약
        weight_decay       = 0.01,
        lr_scheduler_type  = "linear",
        seed               = 42,
        output_dir         = "outputs",
        report_to          = "none",   # wandb 등 외부 로깅 끄기
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# 학습에 걸린 시간과 최대 GPU 메모리 사용량 확인
elapsed = trainer_stats.metrics["train_runtime"]
peak = torch.cuda.max_memory_allocated() / 1024**3

print(f"학습 시간        : {elapsed:.0f}초 ({elapsed/60:.1f}분)")
print(f"최대 GPU 메모리  : {peak:.2f} GB")

# 파인튜닝 후 답변 비교

3단계에서 했던 **같은 질문**을 다시 해봅니다.
KoAlpaca의 한국어 답변 스타일을 학습했기 때문에 답변의 어투나 구성이 달라졌는지 관찰해 보세요.

> 💬 **토론 주제**: 60스텝만 학습했는데도 변화가 보이나요?
> 스텝 수, 데이터 양, 모델 크기를 바꾸면 어떻게 될까요?


In [ ]:
# 추론 모드로 다시 전환
FastLanguageModel.for_inference(model)

In [ ]:
ask("스웨터의 유래에 대해서 알려주세요.")

In [ ]:
ask("인공지능과 머신러닝의 차이를 초등학생에게 설명하듯이 해 주세요.")